In [103]:
# Imports
import pandas as pd
import numpy as np
from tabulate import tabulate
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, BayesianRidge
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings('ignore')

## Utility Functions

In [111]:
def load_node_profile(node):
    filepath = f'../../tool/infrastructure-profiler/profiles/{node}.csv'

    with open(filepath) as f:
        data = f.readlines()[1].split(',')

    return {
        'ram': int(data[1].strip()),
        'cores': int(data[2].strip()),
        'io_read': float(data[3].strip()),
        'io_write': float(data[4].strip()),
        'sysbench': float(data[5].strip()),
        'z7b': float(data[6].strip()),
        'bench_time': float(data[7].strip())
    }

def combine_df(input_df, trace_df):
    rows = []

    for name, group in input_df.groupby('name'):
        no_files = str(len(group))
        filepaths = group['path']
        paths = ','.join([filepath.split('/')[-1].split('.')[-1] for filepath in filepaths])
        total_size = group['size'].sum() / 1073741824  # convert from B to GiB
        task_record = trace_df[trace_df['name'] == name].iloc[0]
        row = [task_record['task_id'], task_record['name'], task_record['%cpu'], 
               task_record['start'], task_record['complete'], task_record['complete'] - task_record['start'], 
               no_files, paths, total_size, task_record['name'].split(':')[-1].split(' ')[0]]
        rows.append(row)

    combined = pd.DataFrame(rows, columns=['id', 'name', '%cpu', 'start', 'complete', 'runtime', 'no_files', 'files', 'input_gib', 'short_name'])
    return combined

def load_profiling_runs(node, workflows, samples, node_profiles):
    node_profile = node_profiles[node]
    runs = {}

    for workflow, samples in zip(workflows, samples):
        runs[workflow] = {}

        for sample in samples:
            sample_dir = f'{node}/{workflow}/{sample}/'
            input_df = pd.read_csv(sample_dir + 'input.csv', sep=';')
            trace_df = pd.read_csv(sample_dir + 'trace.csv')
            runs[workflow][sample] = combine_df(input_df, trace_df)

    return (node_profile, runs)

def load_workflow_execution(cluster, workflow, iteration):
    execution_dir = f'../{cluster}/{workflow}/{iteration}/'
    input_df = pd.read_csv(execution_dir + 'input.csv', sep=';')
    trace_df = pd.read_csv(execution_dir + 'trace.csv')
    return combine_df(input_df, trace_df)

def load_ichnos_trace(cluster, workflow, iteration):
    execution_dir = f'../{cluster}/{workflow}/{iteration}/'
    return pd.read_csv(execution_dir + 'trace.csv')

def load_rapl_summary(cluster, workflow, iteration):
    rapl_vals = pd.read_csv(f'../{cluster}/{workflow}-runs.csv')
    return rapl_vals.iloc[[iteration - 1]]

def load_power_model(node, setting=''):
    if setting != '':
        filename = f'{setting}-model.txt'
    else: 
        filename = 'model.txt'

    with open(f'../../tool/infrastructure-profiler/raw/{node}/{filename}', 'r') as f:
        data = [line.strip() for line in f.readlines()[1:]]

    coeff = float(data[0].split(',')[0][2:].strip())
    inter = float(data[0].split(',')[1][:-2].strip())
    draw = float(data[1].split(':')[1].strip())

    if draw == 0:
        draw = 0.392

    return (coeff, inter, draw)

def get_task_data_from_experiment(task_name, experiment):
    task_data = experiment[experiment['short_name'] == task_name]
    key_data = task_data[['input_gib', 'runtime', '%cpu']]
    key_data.columns = ['data', 'time', 'usage']
    key_data.reset_index().drop('index', axis=1)
    return key_data

In [105]:
class MedianModel:
    def __init__(self, median):
        self.median = median
    
    def predict(self, x):
        x = np.asarray(x)
        value = x.shape[0]
        return np.full(value, self.median)

## Setup Data

In [106]:
# Constants
HU = 'hu'
HU_40 = 'hu-c40'
HU_42 = 'hu-c42'
GPG = 'gpg'
GPG_16 = 'gpgnode-16'
GPG_22 = 'gpgnode-22'
TU = 'tu'
TU_06 = 'tu-06'
AWS = 'aws'
AWS_c5n = 'aws-metal-c5n'
AWS_z1d = 'aws-metal-z1d'
wfs = {
    GPG: ['rnaseq', 'chipseq'], 
    HU: ['rnaseq', 'chipseq', 'rangeland']
}
VR = 'VR'
RF = 'RandomForest'
LR = 'LinearRegression'
BR = 'BayesianRidge'

In [107]:
# Node Profiles + Power Models (todo)
nodes = [AWS_c5n, AWS_z1d, GPG_16, GPG_22, TU_06, HU_40, HU_42] # , 'hu-c44', 'hu-c45']
node_profiles = {}

for node in nodes:
    profile = load_node_profile(node)
    node_profiles[node] = profile

In [109]:
# Workflow Executions 
wf_executions = {}

for cluster, cluster_path in zip([GPG, HU], ['gu-cluster', 'hu-cluster']):
    wf_executions[cluster] = {}
    for wf in wfs[cluster]:
        wf_executions[cluster][wf] = {}
        for iteration in range(1, 4):
            wf_executions[cluster][wf][iteration] = {}
            wf_executions[cluster][wf][iteration]['run'] = load_workflow_execution(cluster_path, wf, iteration)

In [113]:
# Workflow RAPL Total
for cluster, cluster_path in zip([GPG, HU], ['gu-cluster', 'hu-cluster']):
    for wf in wfs[cluster]:
        for iteration in range(1, 4):
            wf_executions[cluster][wf][iteration]['rapl'] = load_rapl_summary(cluster_path, wf, iteration)

In [ ]:
# Load workflow profiling runs
wf_profiles = {}

# TU_06
wfs = ['rnaseq', 'chipseq']
samples = [['sample_b', 'sample_c', 'sample_d'], ['sample_e', 'sample_f', 'sample_g', 'sample_h']]
profile, runs = load_profiling_runs(TU_06, wfs, samples, node_profiles)
wf_profiles[TU_06] = {
    'node_profile': profile,
    'executions': runs,
    'power_model': load_power_model(TU_06)
}

# HU_40
wfs = ['rnaseq', 'chipseq']
samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d', 'sample_e'], ['sample_f', 'sample_g', 'sample_h', 'sample_i', 'sample_j']]
profile, runs = load_profiling_runs(HU_40, wfs, samples, node_profiles)
wf_profiles[HU_40] = {
    'node_profile': profile,
    'executions': runs,
    'power_model': load_power_model(HU_40, 'schedutil')
}

# HU_42
wfs = ['rnaseq', 'chipseq']
samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d', 'sample_e'], ['sample_g', 'sample_h', 'sample_i', 'sample_j']]  # sample_f to be added back
profile, runs = load_profiling_runs(HU_42, wfs, samples, node_profiles)
wf_profiles[HU_42] = {
    'node_profile': profile,
    'executions': runs,
    'power_model': load_power_model(HU_42, 'schedutil')
}

# gpgnode-16
wf_profiles[GPG_16] = {
    'node_profile': node_profiles[GPG_16],
    'power_model': load_power_model(GPG_16, 'ondemand')
}

## Evaluation

In [133]:
# Prediction and Verification
def create_prediction_model(runs, model_type):
    models = {}

    for workflow in runs.keys():
        models[workflow] = {}
        samples = list(runs[workflow].keys())
        tasks = runs[workflow][samples[0]]['short_name'].unique()

        for task in tasks:
            task_data = pd.DataFrame(columns=['data', 'time', 'usage'], dtype='float64')
            for sample in samples: 
                experiment_data = get_task_data_from_experiment(task, runs[workflow][sample])
                task_data = pd.concat([task_data, experiment_data])

            # based off lotaru (what if we have some correlation but not a strong one?)
            # we can also use io_read as a proxy for full input size per task
            pearson = np.corrcoef(task_data['data'].to_numpy(), task_data['time'].to_numpy())[0, 1]
            models[workflow][task] = {}
            models[workflow][task]['usage'] = np.mean(task_data['usage'])

            if np.isnan(pearson) or pearson < 0.75:
               models[workflow][task]['model'] = MedianModel(np.median(task_data['time'].to_numpy()))
               # print('other', pearson, task)
            else:
                if model_type == VR:
                    model = SVR()
                elif model_type == RF:
                    model = RandomForestRegressor()
                elif model_type == BR:
                    model = BayesianRidge(fit_intercept=True)
                else:
                    model = LinearRegression()
                # todo read up more on these regression models

                models[workflow][task]['model'] = model.fit(task_data['data'].values.reshape(-1, 1), task_data['time'].values) 

    return models

def get_translation_factor(local_profile, target_profile):
    local_write_val = local_profile['io_write'] / local_profile['ram']
    local_read_val = local_profile['io_read'] / local_profile['ram']
    target_write_val = target_profile['io_write'] / target_profile['ram']
    target_read_val = target_profile['io_read'] / target_profile['ram']
    return (0.25 * local_profile['z7b'] / target_profile['z7b']) + \
    (0.25 * local_profile['sysbench'] / target_profile['sysbench']) + \
    (0.25 * local_write_val / target_write_val) + \
    (0.25 * local_read_val / target_read_val)
    #(0.25 * local_profile['bench_time'] / target_profile['bench_time']) + \

def predict_wf(wf_model, actual_execution, factor, local_profile, target_profile, target_power_model):
    overall_error = []
    missing_tasks = []    
    headers = ['task', 'input (GiB)', 'pred_time (s)', 'actual_time (s)', 'error (%)', 'pred_usage (%)', 'actual_usage (%)', 'pred_energy (kWh)', 'actual_energy (kWh)', 'error (%)']
    data = []

    for _, record in actual_execution.iterrows():
        task_name = record['short_name']
        data_size = record['input_gib']
        actual_runtime = record['runtime']

        if task_name in wf_model.keys():
            if local_profile['cores'] != target_profile['cores']:
                predicted_runtime = wf_model[task_name]['model'].predict([[data_size]])[0] * factor * (wf_model[task_name]['usage'] / target_profile['cores'])
                pred_usage = (pred_usage / local_profile['cores']) * target_profile['cores']
            else:
                predicted_runtime = wf_model[task_name]['model'].predict([[data_size]])[0] * factor  

            pred_usage = wf_model[task_name]['usage']

            if actual_runtime > 0:
                if predicted_runtime > 0:
                    error = (abs(predicted_runtime - actual_runtime) / actual_runtime) * 100 
                    overall_error.append(error)

            data.append([task_name, f'{data_size:.2f}', f'{predicted_runtime / 1000:.2f}', f'{actual_runtime / 1000:.2f}', error, f'{pred_usage:.2f}', f'{record["%cpu"]}', 'todo', 'todo', 'todo'])
        else:
            missing_tasks.append(task_name)  # what to do if the task doesn't exist from the profiling runs? 
            # come up with a backup model that takes the average of all either median tasks? 
            # can we choose tasks with a similar name? 

    print(tabulate(data, headers, tablefmt='orgtbl'))
    print('missing:', missing_tasks)
    print('avg. prediction error:', np.mean(overall_error))

    # predict task dynamic energy consumption
    # predict workflow total energy consumption (include static energy during times when nodes were individually active)

    # load task dynamic energy consumption (estimated using ichnos)
    # load workflow total energy consumption (estimated from RAPL energy logs -- ground truth here)

    # compare the error % for task dynamic energy consumption
    # compare the error % in total energy consumption

In [134]:
# TEST: hu_c40 to hu-cluster execution
hu_wf_models = create_prediction_model(wf_profiles[HU_40]['executions'], LR)
local_profile = wf_profiles[HU_40]['node_profile']
target_profile = wf_profiles[HU_40]['node_profile']
factor = get_translation_factor(local_profile, target_profile)
target_power_model = wf_profiles[HU_40]['power_model']
hu_rnaseq_model = hu_wf_models['rnaseq']
actual_execution = wf_executions[HU]['rnaseq'][1]['run']
predict_wf(hu_rnaseq_model, actual_execution, factor, local_profile, target_profile, target_power_model)

| task                     |   input (GiB) |   pred_time (s) |   actual_time (s) |   error (%) |   pred_usage (%) |   actual_usage (%) | pred_energy (kWh)   | actual_energy (kWh)   | error (%)   |
|--------------------------+---------------+-----------------+-------------------+-------------+------------------+--------------------+---------------------+-----------------------+-------------|
| SAMTOOLS_FLAGSTAT        |          1.58 |           13.74 |                14 |   1.82186   |           137.09 |              132.8 | todo                | todo                  | todo        |
| SAMTOOLS_FLAGSTAT        |          1.58 |           13.74 |                14 |   1.82157   |           137.09 |              133.5 | todo                | todo                  | todo        |
| SAMTOOLS_FLAGSTAT        |          1.58 |           13.74 |                16 |  14.0956    |           137.09 |              133.5 | todo                | todo                  | todo        |
| SAMTOOLS_FLAG

In [135]:
# TEST: hu_c40 model to gu-cluster execution
local_profile = wf_profiles[HU_40]['node_profile']
target_profile = wf_profiles[GPG_16]['node_profile']
factor = get_translation_factor(local_profile, target_profile)
target_power_model = wf_profiles[GPG_16]['power_model']
hu_rnaseq_model = hu_wf_models['rnaseq']
actual_execution = wf_executions[GPG]['rnaseq'][1]['run']
predict_wf(hu_rnaseq_model, actual_execution, factor, local_profile, target_profile, target_power_model)

| task                     |   input (GiB) |   pred_time (s) |   actual_time (s) |   error (%) |   pred_usage (%) |   actual_usage (%) | pred_energy (kWh)   | actual_energy (kWh)   | error (%)   |
|--------------------------+---------------+-----------------+-------------------+-------------+------------------+--------------------+---------------------+-----------------------+-------------|
| CUSTOM_GETCHROMSIZES     |          2.93 |           23.17 |                19 |    21.9242  |           100.06 |               60.9 | todo                | todo                  | todo        |
| GTF_FILTER               |          3.78 |           42.07 |                30 |    40.2334  |            99.92 |               79.4 | todo                | todo                  | todo        |
| MAKE_TRANSCRIPTS_FASTA   |          3.71 |          120.13 |                96 |    25.1323  |           100.08 |               76.3 | todo                | todo                  | todo        |
| SALMON_INDEX 

In [ ]:
# THE PLAN

# also - look at tasks that don't scale linearly (pearsson)
# investigate tasks with prediction of negative runtime?
# investigate tasks with runtime of 0

# prediction error for runtime should be ~15%, not up at 40/50%

# confirm on whether comparing dynamic power consumption for individual tasks, and then overall should consider static over periods of active execution (using traces)

# identify the governor and compare energy estimation for each workflow using ichnos (!)
# have these task-level estimates of power consumed, and total inc. and exc. static power consumption for the nodes
# automate the process

# next step, combine the predicted runtime with estimation of energy using node power models
# start to compare task-level estimates with traces


In [ ]:
# mse = mean_squared_error(y_test, y_pred) 
# print(f'Linear Regression MSE: {mse}')